# 07 · Investigation: from an incident to its likely cause

Phase 8, Step 8.6. Every incident gets an evidence pack, a where/when analysis, a template
explanation from `configs/playbooks.yml`, suggested actions (pending approval) and a report.
This notebook shows them for the pipeline's incidents, one incident in depth, and how often the
explainer names the right cause on the fault experiments.

**Before running:**
```
python src/gold/rebuild.py                    # includes the investigate stage
python src/evaluation/run_experiments.py      # for section 7 (optional)
python src/evaluation/explanation_accuracy.py
```
The last section prints a verdict to paste back.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

from common import db
from common.config import load_datasets_config
from evaluation import explanation_accuracy as xa
from evaluation import fault_injector as fi

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)
INK, SURFACE = "#52514e", "#fcfcfb"
CONFIDENCE = {"high": "#1d5fb4", "medium": "#3987e5", "low": "#a9cbf3"}
OUTCOME = {"top1": "#0ca30c", "top3": "#fab219", "wrong": "#d03b3b", "unknown": "#8a8a85",
           "not_detected": "#d9d8d4"}

## Settings

In [ ]:
DATABASE = Path(os.environ.get("DATAGUARD_DATABASE", load_datasets_config().paths["duckdb"]))
con = db.connect(DATABASE, read_only=True)
investigations = con.execute(
    """
    SELECT v.incident_id, i.title, i.severity, i.status, v.period, v.playbook, v.explained,
           v.top_cause, v.confidence, v.headline, v.where_verdict, v.explanation, v.report_path
    FROM investigations v JOIN incidents i USING (incident_id) ORDER BY v.incident_id
    """
).df()
print(f"{len(investigations)} incident(s) investigated in {DATABASE.name}")
if investigations.empty:
    print("nothing to show yet: run python src/investigation/investigate.py")

## 1 · Every incident and its likely cause

The best cause of each incident's playbook, its confidence, and the where/when answer. "Cause
unknown" means no cause reached medium confidence.

In [ ]:
overview = investigations[["incident_id", "title", "severity", "status", "period", "top_cause",
                           "confidence", "explained", "where_verdict"]]
overview

## 2 · One incident in depth

The most severe active incident (or `DATAGUARD_INCIDENT`): its ranked causes with the evidence
for each, and its full report.

In [ ]:
RANK = {"critical": 0, "high": 1, "medium": 2, "low": 3}
chosen = os.environ.get("DATAGUARD_INCIDENT")
if not chosen and not investigations.empty:
    pick = investigations.assign(
        active=investigations["status"] != "RESOLVED",
        rank=investigations["severity"].map(RANK),
    ).sort_values(["active", "rank", "incident_id"], ascending=[False, True, True])
    chosen = pick.iloc[0]["incident_id"]
row = investigations.set_index("incident_id").loc[chosen] if chosen else None
if row is not None:
    explanation = json.loads(row["explanation"])
    print(explanation["headline"])
    print(f"where/when: {explanation['where']}")
    display(pd.DataFrame([
        {"rank": i, "cause": c["cause_id"], "confidence": c["confidence"], "score": c["score"],
         "text": c["text"], "evidence": "; ".join(c["evidence"])}
        for i, c in enumerate(explanation["causes"], 1)
    ]))

In [ ]:
if row is not None and row["report_path"]:
    report = Path(row["report_path"])
    report = report if report.is_absolute() else REPO_ROOT / report
    if report.is_file():
        display(Markdown(report.read_text(encoding="utf-8")))
    else:
        print(f"report not found: {report}")

## 3 · Causes and confidence

In [ ]:
counts = (investigations.assign(top_cause=investigations["top_cause"].fillna("(none)"),
                                confidence=investigations["confidence"].fillna("low"))
          .groupby(["top_cause", "confidence"]).size().unstack(fill_value=0))
fig = go.Figure()
for level in ("high", "medium", "low"):
    if level in counts:
        fig.add_bar(y=counts.index, x=counts[level], name=level, orientation="h",
                    marker_color=CONFIDENCE[level],
                    hovertemplate="%{y}: %{x} incident(s), " + level + "<extra></extra>")
fig.update_layout(barmode="stack", title="Best cause per incident, by confidence",
                  xaxis_title="incidents", plot_bgcolor=SURFACE,
                  height=max(260, 40 * len(counts) + 120), legend_title_text="confidence",
                  legend_traceorder="normal", yaxis_autorange="reversed",
                  xaxis=dict(dtick=1, tickformat="d"))
fig.show()

## 4 · Where and when

For row-level incidents, the segments that hold the affected rows at a clearly higher rate
(`configs/investigation.yml`). An empty table means no incident's problem is concentrated: the
explainer then claims no place.

In [ ]:
packs = {i: json.loads(e) for i, e in con.execute(
    "SELECT incident_id, evidence FROM investigation_evidence ORDER BY 1").fetchall()}
where_rows = []
for incident_id, pack in packs.items():
    for f in (pack.get("concentration") or {}).get("findings") or []:
        where_rows.append({"incident_id": incident_id, **f})
where = pd.DataFrame(where_rows)
print(f"{len(where)} concentration(s) found in {len(packs)} evidence pack(s)")
where

## 5 · Suggested actions\n\nNothing runs them; a person decides with `investigate.py --decide`.

In [ ]:
actions = con.execute(
    "SELECT incident_id, action_id, kind, status, decided_by, text FROM investigation_actions "
    "ORDER BY incident_id, action_id"
).df()
print(actions["status"].value_counts().to_dict() if not actions.empty else "no actions")
actions

## 6 · Explanation accuracy on the fault experiments

Each fault in `configs/faults.yml` names the causes that count as right and where the problem
sits. Accuracy counts the experiments a detector caught.

In [ ]:
faults_path = os.environ.get("DATAGUARD_FAULTS_CONFIG")
faults = fi.load_faults_config(Path(faults_path) if faults_path else None)
try:
    judged = xa.evaluate_explanations(faults)
    accuracy = xa.accuracy(judged)
except FileNotFoundError as exc:
    judged, accuracy = None, None
    print(f"no experiments yet: {exc}")
if judged is not None:
    print(json.dumps(accuracy, indent=2))
    display(judged[["experiment_id", "detector", "top_cause", "confidence", "outcome",
                    "where_claimed", "where_outcome"]])

In [ ]:
if judged is not None and not judged.empty:
    table = judged.groupby(["fault_id", "outcome"]).size().unstack(fill_value=0)
    fig = go.Figure()
    for outcome in xa.OUTCOMES:
        if outcome in table:
            fig.add_bar(y=table.index, x=table[outcome], name=outcome, orientation="h",
                        marker_color=OUTCOME[outcome],
                        hovertemplate="%{y}: %{x} experiment(s) " + outcome + "<extra></extra>")
    fig.update_layout(barmode="stack", title="Explanations per fault", xaxis_title="experiments",
                      plot_bgcolor=SURFACE, height=max(260, 32 * len(table) + 120),
                      legend_title_text="explanation", legend_traceorder="normal",
                      yaxis_autorange="reversed", xaxis=dict(dtick=1, tickformat="d"))
    fig.show()

## 7 · Verdict\n\nCopy the printed text back.

In [ ]:
verdict = {
    "incidents": len(investigations),
    "explained": int(investigations["explained"].sum()),
    "cause_unknown": investigations.loc[~investigations["explained"], "incident_id"].tolist(),
    "top_causes": dict(zip(investigations["incident_id"], investigations["top_cause"])),
    "confidence": investigations["confidence"].value_counts().to_dict(),
    "concentrations": [f"{r.incident_id}: {r.dimension} = {r.value}" for r in where.itertuples()],
    "actions": actions["status"].value_counts().to_dict() if not actions.empty else {},
}
if accuracy is not None:
    verdict["explanation_accuracy"] = {
        k: accuracy[k] for k in ("detected", "top1", "top3", "top1_accuracy", "top3_accuracy",
                                 "wrong", "unknown", "false_where", "where_found", "where_missed")
    }
print(json.dumps(verdict, indent=2, default=str))

In [ ]:
con.close()